In [2]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT))


# FECFIN - CONTA AZUL

In [23]:
import shutil
import re
import pandas as pd
from pathlib import Path
from src.utils.helpers import totalizador, planilha_lancamento

arquivo = Path(r"G:\.shortcut-targets-by-id\1nboB01tRteAh3IDImIuSVc8ksH6cRrEH\PROJ EXT\460_LIVRARIA WR\Extratos\26\05\0526_FECFIN_LIVRARIA WR.xls")

origem_lanc = Path(r"C:\Users\manja\OneDrive\Documentos\AutoExtrato\data\Lancamentos_Contabeis.xlsm")
destino_lanc = Path(r"G:\.shortcut-targets-by-id\1nboB01tRteAh3IDImIuSVc8ksH6cRrEH\PROJ EXT\460_LIVRARIA WR\Extratos\26\05")

df = pd.read_excel(arquivo)
df["DESCRIÇÃO"] = df.apply(lambda x: f"{x["Descrição"]} {x["Nome do fornecedor/cliente"]}", axis=1)
df = df[["Data movimento", "DESCRIÇÃO", "Valor (R$)", "Conta bancária"]]
df["DESCRIÇÃO"] = (df["DESCRIÇÃO"].str.strip().str.replace("nan", "").str.upper())
df["TIPO"] = df.apply(lambda x: "C" if x["Valor (R$)"] > 0 else "D", axis=1)
df = df.rename(columns={"Data movimento": "DATA", "Valor (R$)": "VALOR", "Conta bancária": "BANCO"})
df["VALOR"] = df["VALOR"].abs()
dfs_por_conta = {
    conta: grupo.copy()
    for conta, grupo in df.groupby("BANCO")
}

# Gera um arquivo para cada banco
for banco, df_banco in dfs_por_conta.items():
    
    # Limpa o nome do banco para usar no nome do arquivo
    nome_banco = str(banco)
    
    # Nome do arquivo final
    arquivo_lanc = destino_lanc / f"[LANC] {arquivo.stem} - {nome_banco}.xlsm"

    # Copia o modelo original
    shutil.copy2(origem_lanc, arquivo_lanc)

    # Remove a coluna BANCO se ela não for necessária na planilha de lançamento
    df_lancamento = df_banco.drop(columns=["BANCO"])

    # Preenche a planilha
    planilha_lancamento(df_lancamento, arquivo_lanc)
    

c:\Users\manja\OneDrive\Documentos\AutoExtrato\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Arquivo preenchido com sucesso: G:\.shortcut-targets-by-id\1nboB01tRteAh3IDImIuSVc8ksH6cRrEH\PROJ EXT\460_LIVRARIA WR\Extratos\26\05\[LANC] 0526_FECFIN_LIVRARIA WR - Sicoob - Livraria.xlsm


# FECFIN

In [26]:
import shutil
import pandas as pd
from pathlib import Path
from src.utils.helpers import totalizador, planilha_lancamento

origem_lanc = Path(r"C:\Users\manja\OneDrive\Documentos\AutoExtrato\data\Lancamentos_Contabeis.xlsm")
destino_lanc = Path(r"G:\.shortcut-targets-by-id\1nboB01tRteAh3IDImIuSVc8ksH6cRrEH\PROJ EXT\156_CEMAF OPE\Extratos\26\04")
arquivo = Path(r"G:\.shortcut-targets-by-id\1nboB01tRteAh3IDImIuSVc8ksH6cRrEH\PROJ EXT\156_CEMAF OPE\Extratos\26\04\0426_FECFIN_CEMAF OPE.xlsx")

with pd.ExcelFile(arquivo) as xls:
    abas = xls.sheet_names

    if "CEMAF" in arquivo.stem:
        bancos = ["UNICRED", "SICOOB", "SICREDI", "CAIXA"]
        extratos = [aba for aba in abas for banco in bancos if banco in aba]

        for extrato in extratos:
            df = pd.read_excel(xls, sheet_name=extrato)

            if "UNICRED" in extrato:
                df.columns = df.iloc[5]
                df = df[6:].reset_index(drop=True)

                df["DESCRIÇÃO"] = df.apply(
                    lambda x: f'{x["HISTÓRICO"]} {x["OBS"]} {x["OBS P/ INTERNAS"]} {x["TIPO"]} {x["Nº DOC"]}',
                    axis=1
                )

                df["DESCRIÇÃO"] = df["DESCRIÇÃO"].str.replace("nan", "", regex=False).str.upper()
                df = df[["DATA", "DESCRIÇÃO", "ENTRADA", "SAIDA", "SALDO"]]

                df["ENTRADA"] = pd.to_numeric(df["ENTRADA"], errors="coerce").fillna(0)
                df["SAIDA"] = pd.to_numeric(df["SAIDA"], errors="coerce").fillna(0)

                df["VALOR"] = df["ENTRADA"].where(df["ENTRADA"] != 0, df["SAIDA"] * -1)
                df = df.loc[~df["DESCRIÇÃO"].astype(str).str.upper().str.contains("SALDO", na=False)]

                df["TIPO"] = df["VALOR"].apply(lambda valor: "C" if valor > 0 else "D")
                df = df[["DATA", "DESCRIÇÃO", "VALOR", "TIPO"]]
                df["VALOR"] = df["VALOR"].abs()

            elif "SICOOB" in extrato:
                df.columns = df.iloc[5]
                df = df[6:].reset_index(drop=True)

                df["DESCRIÇÃO"] = df.apply(
                    lambda x: f'{x["HISTÓRICO"]} {x["OBS P/ CONT"]} {x["OBS P/ INTERNAS"]} {x["TIPO"]} {x["Nº DOC"]}',
                    axis=1
                )

                df["DESCRIÇÃO"] = df["DESCRIÇÃO"].str.replace("nan", "", regex=False).str.upper()
                df = df[["DATA", "DESCRIÇÃO", "ENTRADA", "SAIDA", "SALDO"]]
                df = df.dropna(subset=["DATA"])

                df["ENTRADA"] = pd.to_numeric(df["ENTRADA"], errors="coerce").fillna(0)
                df["SAIDA"] = pd.to_numeric(df["SAIDA"], errors="coerce").fillna(0)

                df["VALOR"] = df["ENTRADA"].where(df["ENTRADA"] != 0, df["SAIDA"] * -1)
                df = df.loc[~df["DESCRIÇÃO"].astype(str).str.upper().str.contains("SALDO", na=False)]

                df["TIPO"] = df["VALOR"].apply(lambda valor: "C" if valor > 0 else "D")
                df = df[["DATA", "DESCRIÇÃO", "VALOR", "TIPO"]]
                df["VALOR"] = df["VALOR"].abs()

            elif "SICREDI" in extrato:
                df.columns = df.iloc[6]
                df = df[7:].reset_index(drop=True)

                df["DESCRIÇÃO"] = df.apply(
                    lambda x: f'{x["HISTÓRICO"]} {x["OBS"]} {x["TIPO"]} {x["Nº DOC"]}',
                    axis=1
                )

                df["DESCRIÇÃO"] = df["DESCRIÇÃO"].str.replace("nan", "", regex=False).str.upper()
                df = df[["DATA", "DESCRIÇÃO", "ENTRADA", "SAIDA", "SALDO"]]

                df["ENTRADA"] = pd.to_numeric(df["ENTRADA"], errors="coerce").fillna(0)
                df["SAIDA"] = pd.to_numeric(df["SAIDA"], errors="coerce").fillna(0)

                df["VALOR"] = df["ENTRADA"].where(df["ENTRADA"] != 0, df["SAIDA"] * -1)
                df = df.loc[~df["DESCRIÇÃO"].astype(str).str.upper().str.contains("SALDO", na=False)]

                df["TIPO"] = df["VALOR"].apply(lambda valor: "C" if valor > 0 else "D")
                df = df[["DATA", "DESCRIÇÃO", "VALOR", "TIPO"]]
                df["VALOR"] = df["VALOR"].abs()

            elif "CAIXA" in extrato:
                df.columns = df.iloc[4]
                df = df[5:].reset_index(drop=True)

                df["DESCRIÇÃO"] = df.apply(
                    lambda x: f'{x["HISTÓRICO"]} {x["OBS"]} {x["DADOS BANCÁRIOS"]} {x["TIPO"]} {x["Nº DOC"]}',
                    axis=1
                )

                df["DESCRIÇÃO"] = df["DESCRIÇÃO"].str.replace("nan", "", regex=False).str.upper()
                df = df[["DATA", "DESCRIÇÃO", "ENTRADA", "SAIDA", "SALDO"]]

                df["ENTRADA"] = pd.to_numeric(df["ENTRADA"], errors="coerce").fillna(0)
                df["SAIDA"] = pd.to_numeric(df["SAIDA"], errors="coerce").fillna(0)

                df["VALOR"] = df["ENTRADA"].where(df["ENTRADA"] != 0, df["SAIDA"] * -1)
                df = df.loc[~df["DESCRIÇÃO"].astype(str).str.upper().str.contains("SALDO", na=False)]

                df["TIPO"] = df["VALOR"].apply(lambda valor: "C" if valor > 0 else "D")
                df = df[["DATA", "DESCRIÇÃO", "VALOR", "TIPO"]]
                df["VALOR"] = df["VALOR"].abs()

                df["DATA"] = pd.to_datetime(
                    df["DATA"],
                    errors="coerce"
                ).dt.strftime("%d/%m/%Y")

            else:
                continue

            arquivo_lanc = destino_lanc / f"[LANC] {arquivo.stem}_{extrato}.xlsm"

            shutil.copy2(origem_lanc, arquivo_lanc)

            planilha_lancamento(df, arquivo_lanc)



Arquivo preenchido com sucesso: G:\.shortcut-targets-by-id\1nboB01tRteAh3IDImIuSVc8ksH6cRrEH\PROJ EXT\156_CEMAF OPE\Extratos\26\04\[LANC] 0426_FECFIN_CEMAF OPE_UNICRED 10378-0.xlsm
Arquivo preenchido com sucesso: G:\.shortcut-targets-by-id\1nboB01tRteAh3IDImIuSVc8ksH6cRrEH\PROJ EXT\156_CEMAF OPE\Extratos\26\04\[LANC] 0426_FECFIN_CEMAF OPE_SICOOB 18723-2.xlsm
Arquivo preenchido com sucesso: G:\.shortcut-targets-by-id\1nboB01tRteAh3IDImIuSVc8ksH6cRrEH\PROJ EXT\156_CEMAF OPE\Extratos\26\04\[LANC] 0426_FECFIN_CEMAF OPE_SICREDI.xlsm
Arquivo preenchido com sucesso: G:\.shortcut-targets-by-id\1nboB01tRteAh3IDImIuSVc8ksH6cRrEH\PROJ EXT\156_CEMAF OPE\Extratos\26\04\[LANC] 0426_FECFIN_CEMAF OPE_CAIXA.xlsm
